In [1]:

from pathlib import Path
import pandas as pd
import numpy as np

import config
from etl.data_loader import DataLoader

loader = DataLoader()

print("DATA_ROOT:", config.PathConfig.DATA_ROOT)
print("PROCESSED:", config.PathConfig.PROCESSED)
print("RAW:", config.PathConfig.RAW)

DATA_ROOT: D:\work and study\PostGraduate\HK\project\TradingSystem\data_storage
PROCESSED: D:\work and study\PostGraduate\HK\project\TradingSystem\data_storage\processed
RAW: D:\work and study\PostGraduate\HK\project\TradingSystem\data_storage\raw


In [15]:
processed_files = sorted(Path(config.PathConfig.PROCESSED).glob("*.parquet"))

pd.DataFrame({
    "file": [p.name for p in processed_files],
    "path": [str(p) for p in processed_files],
    "size_mb": [round(p.stat().st_size / 1024 / 1024, 2) for p in processed_files],
})

,file,path,size_mb
0,BNBUSDT_1d.parquet,D:\work and study\PostGraduate\HK\project\Trad...,0.09
1,BNBUSDT_1h.parquet,D:\work and study\PostGraduate\HK\project\Trad...,2.20
2,BNBUSDT_1m.parquet,D:\work and study\PostGraduate\HK\project\Trad...,75.01
3,BNBUSDT_4h.parquet,D:\work and study\PostGraduate\HK\project\Trad...,0.55
4,BTCUSDT_1d.parquet,D:\work and study\PostGraduate\HK\project\Trad...,0.10
5,BTCUSDT_1h.parquet,D:\work and study\PostGraduate\HK\project\Trad...,2.65
6,BTCUSDT_1m.parquet,D:\work and study\PostGraduate\HK\project\Trad...,90.24
7,BTCUSDT_4h.parquet,D:\work and study\PostGraduate\HK\project\Trad...,0.61
8,ETHUSDT_1d.parquet,D:\work and study\PostGraduate\HK\project\Trad...,0.10
9,ETHUSDT_1h.parquet,D:\work and study\PostGraduate\HK\project\Trad...,2.52


In [3]:
symbol = "BTC/USDT"
timeframe = "4h"

df = loader.get_crypto_kline_data(
    symbol=symbol,
    timeframe=timeframe,
)

df.head()

,high,net_taker_vol,volume,low,taker_buy_vol,open,close
timestamp,,,,,,,
2021-01-01 00:00:00,29546.42,495.037,43210.161,28706.00,21852.599,28948.19,29302.11
2021-01-01 04:00:00,29422.32,-3136.344,26682.086,28822.00,11772.871,29302.11,29107.71
2021-01-01 08:00:00,29454.45,-339.046,29562.630,28900.00,14611.792,29107.72,29341.99
2021-01-01 12:00:00,29668.86,-1227.234,49142.952,29043.75,23957.859,29342.00,29210.84
2021-01-01 16:00:00,29388.10,-3641.582,43668.170,28627.12,20013.294,29210.85,29048.47


In [17]:
def describe_df(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "column": df.columns,
        "dtype": [str(df[c].dtype) for c in df.columns],
        "non_null": [df[c].notna().sum() for c in df.columns],
        "missing": [df[c].isna().sum() for c in df.columns],
        "missing_pct": [df[c].isna().mean() for c in df.columns],
        "sample_value": [df[c].dropna().iloc[0] if df[c].dropna().shape[0] else None for c in df.columns],
    })

describe_df(df)

,column,dtype,non_null,missing,missing_pct,sample_value
0,taker_buy_vol,float64,11924,0,0.0,21852.599
1,net_taker_vol,float64,11924,0,0.0,495.037
2,close,float64,11924,0,0.0,29302.110
3,volume,float64,11924,0,0.0,43210.161
4,high,float64,11924,0,0.0,29546.420
5,open,float64,11924,0,0.0,28948.190
6,low,float64,11924,0,0.0,28706.000


In [18]:
pd.DataFrame({
    "symbol": [symbol],
    "timeframe": [timeframe],
    "start": [df.index.min()],
    "end": [df.index.max()],
    "rows": [len(df)],
    "columns": [list(df.columns)],
})

,symbol,timeframe,start,end,rows,columns
0,BTC/USDT,4h,2021-01-01,2026-06-11 04:00:00,11924,"[taker_buy_vol, net_taker_vol, close, volume, ..."


In [24]:
matrix = loader.get_crypto_matrix(
    symbols=["BTC/USDT", "ETH/USDT", "SOL/USDT", "BNB/USDT"],
    timeframe="4h",
    columns=["close", "volume", "net_taker_vol"],
)

close = matrix["volume"]
close.tail()

读取加密货币数据 (周期: 4h)...
✅ 成功加载 3 个特征矩阵。


,BTC/USDT,ETH/USDT,SOL/USDT,BNB/USDT
timestamp,,,,
2026-06-09 12:00:00,71792.227,1624864.053,7920044.95,168656.41
2026-06-09 16:00:00,51223.379,1356989.394,6688645.28,100301.68
2026-06-09 20:00:00,17544.777,513326.636,2057497.20,51072.06
2026-06-10 00:00:00,19223.281,565567.199,2573453.63,64341.82
2026-06-10 04:00:00,19399.655,539454.290,2311561.68,81668.74


In [6]:
#资金费率表格查询
fund_rate = loader.get_funding_rate_data()
raw_rate = loader.get_raw_funding_rate_data("BTC/USDT")
display(raw_rate)

,symbol,funding_rate,source,created_at
timestamp,,,,
2021-01-01 00:00:00.002,BTC/USDT,0.000228,binance_usdm,2026-06-11 08:24:37.968379
2021-01-01 08:00:00.006,BTC/USDT,0.000263,binance_usdm,2026-06-11 08:24:37.968379
2021-01-01 16:00:00.003,BTC/USDT,0.000345,binance_usdm,2026-06-11 08:24:37.968379
2021-01-02 00:00:00.000,BTC/USDT,0.000100,binance_usdm,2026-06-11 08:24:37.968379
2021-01-02 08:00:00.000,BTC/USDT,0.000202,binance_usdm,2026-06-11 08:24:37.968379
...,...,...,...,...
2026-06-10 00:00:00.001,BTC/USDT,0.000004,binance_usdm,2026-06-11 08:24:37.968379
2026-06-10 08:00:00.011,BTC/USDT,-0.000034,binance_usdm,2026-06-11 08:24:37.968379
2026-06-10 16:00:00.009,BTC/USDT,0.000025,binance_usdm,2026-06-11 08:24:37.968379


In [5]:
from etl.feature_builder_H import build_crypto_features, FeatureBuilderConfig
from etl.feature_registry import get_feature_definitions


cfg = FeatureBuilderConfig(
    decision_timeframe="4h",
    include_funding=True,
    include_oi=True,
    include_cvd_proxy=True,
    include_sentiment=True,
    include_onchain=True,
)

# features = build_crypto_features(cfg=cfg, save=True)
# display(features)
defs = get_feature_definitions()
display(defs)

,feature,group,definition,calculation,source,usage
0,symbol,identity,"Trading pair, e.g. BTC/USDT.",Copied from config.TargetConfig.COINS / source...,config / processed market files,Primary entity key. Not a numeric model feature.
1,ts_open,time_grid,Open timestamp of the 4h decision bar.,"processed 4h bar timestamp. With label='left',...",processed/{SYMBOL}_4h.parquet.timestamp,Audit / explainability. Not a model feature by...
2,ts_close,time_grid,Close timestamp of the 4h decision bar.,ts_open + 4h.,derived from ts_open,Defines when the 4h bar has completed.
3,decision_time,time_grid,Time at which the strategy/Agent is allowed to...,ts_close + cfg.market_latency; default = ts_cl...,derived,Primary PIT merge key. All features must satis...
4,open_4h,market_4h,Open price of the completed 4h bar.,first 1m open within the 4h resample window.,processed/{SYMBOL}_4h.parquet.open,4h price structure.
5,high_4h,market_4h,High price of the completed 4h bar.,max 1m high within the 4h resample window.,processed/{SYMBOL}_4h.parquet.high,4h price range / volatility proxy.
6,low_4h,market_4h,Low price of the completed 4h bar.,min 1m low within the 4h resample window.,processed/{SYMBOL}_4h.parquet.low,4h price range / volatility proxy.
7,close_4h,market_4h,Close price of the completed 4h bar.,last 1m close within the 4h resample window.,processed/{SYMBOL}_4h.parquet.close,Main price anchor for returns and downstream b...
8,volume_4h,market_4h,Total traded volume in the completed 4h bar.,sum of 1m volume within the 4h resample window.,processed/{SYMBOL}_4h.parquet.volume,Liquidity / activity state.
9,ret_4h,market_4h,Most recent 4h return.,close_4h.pct_change(1).,derived from close_4h,Short momentum / immediate price change.


In [2]:
# On-chain / DeFi data preview
def describe_onchain_df(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "column": df.columns,
        "dtype": [str(df[c].dtype) for c in df.columns],
        "non_null": [df[c].notna().sum() for c in df.columns],
        "missing": [df[c].isna().sum() for c in df.columns],
        "missing_pct": [round(df[c].isna().mean() * 100, 2) for c in df.columns],
        "sample_value": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })

onchain_tables_processed = loader.list_onchain_tables(layer="processed")
onchain_tables_raw = loader.list_onchain_tables(layer="raw")
onchain_tables_factors = loader.list_onchain_tables(layer="factors")

display(onchain_tables_processed)
display(onchain_tables_raw)
display(onchain_tables_factors)

onchain_daily = loader.get_onchain_data(table="onchain_daily", layer="processed")
onchain_features = loader.get_onchain_data(table="onchain_features", layer="factors")
display(onchain_daily.tail())
display(onchain_features.tail() if onchain_features is not None else None)

if onchain_daily is not None and not onchain_daily.empty:
    display(describe_onchain_df(onchain_daily.reset_index()))
    display(loader.get_latest_onchain_row(table="onchain_daily", layer="processed"))

if onchain_features is not None and not onchain_features.empty:
    display(describe_onchain_df(onchain_features.reset_index()))
    display(loader.get_latest_onchain_row(table="onchain_features", layer="factors"))


,layer,source,table,path,size_mb,modified_at
0,processed,processed,defillama_daily,D:\work and study\PostGraduate\HK\project\Trad...,0.1692,2026-06-12 15:56:57.180091
1,processed,processed,onchain_daily,D:\work and study\PostGraduate\HK\project\Trad...,0.1692,2026-06-12 15:56:57.289846


,layer,source,table,path,size_mb,modified_at
0,raw,defillama,chain_tvl_bitcoin,D:\work and study\PostGraduate\HK\project\Trad...,0.0283,2026-06-12 15:10:53.408072
1,raw,defillama,chain_tvl_bsc,D:\work and study\PostGraduate\HK\project\Trad...,0.0371,2026-06-12 15:11:00.764991
2,raw,defillama,chain_tvl_ethereum,D:\work and study\PostGraduate\HK\project\Trad...,0.0553,2026-06-12 15:10:55.953797
3,raw,defillama,chain_tvl_solana,D:\work and study\PostGraduate\HK\project\Trad...,0.0292,2026-06-12 15:10:58.307781
4,raw,defillama,dex_volume_global,D:\work and study\PostGraduate\HK\project\Trad...,0.0497,2026-06-12 15:11:13.232873
5,raw,defillama,fees_revenue_global,D:\work and study\PostGraduate\HK\project\Trad...,0.0377,2026-06-12 15:11:22.029221
6,raw,defillama,stablecoins_all,D:\work and study\PostGraduate\HK\project\Trad...,0.0542,2026-06-12 15:11:05.904797


,layer,source,table,path,size_mb,modified_at
0,factors,factors,onchain_features,D:\work and study\PostGraduate\HK\project\Trad...,0.7411,2026-06-12 15:56:57.382441


,onchain_defillama_tvl_bitcoin_usd,onchain_defillama_tvl_bsc_usd,onchain_defillama_tvl_ethereum_usd,onchain_defillama_tvl_solana_usd,onchain_defillama_selected_chains_tvl_usd,onchain_defillama_stablecoin_mcap_usd,onchain_defillama_dex_volume_usd,onchain_defillama_fees_usd
timestamp,,,,,,,,
2026-06-08,4.198599e+09,5.205459e+09,3.768732e+10,4.925316e+09,5.201669e+10,3.141817e+11,7.224427e+09,56658058.0
2026-06-09,4.129998e+09,5.175929e+09,3.750249e+10,4.877264e+09,5.168568e+10,3.147107e+11,6.844394e+09,51492069.0
2026-06-10,4.061126e+09,5.162583e+09,3.686146e+10,4.781674e+09,5.086684e+10,3.136835e+11,6.709375e+09,53074285.0
2026-06-11,4.070036e+09,5.156885e+09,3.655583e+10,4.438993e+09,5.022175e+10,3.136407e+11,5.964061e+09,54239853.0
2026-06-12,4.178760e+09,5.210223e+09,3.729126e+10,4.633518e+09,5.131376e+10,3.133113e+11,5.785529e+09,54492695.0


,onchain_defillama_tvl_bitcoin_usd,onchain_defillama_tvl_bsc_usd,onchain_defillama_tvl_ethereum_usd,onchain_defillama_tvl_solana_usd,onchain_defillama_selected_chains_tvl_usd,onchain_defillama_stablecoin_mcap_usd,onchain_defillama_dex_volume_usd,onchain_defillama_fees_usd,onchain_defillama_tvl_bitcoin_usd_chg_1d,onchain_defillama_tvl_bitcoin_usd_chg_7d,...,onchain_defillama_selected_chains_tvl_usd_z_30d,onchain_defillama_stablecoin_mcap_usd_chg_1d,onchain_defillama_stablecoin_mcap_usd_chg_7d,onchain_defillama_stablecoin_mcap_usd_z_30d,onchain_defillama_dex_volume_usd_chg_1d,onchain_defillama_dex_volume_usd_chg_7d,onchain_defillama_dex_volume_usd_z_30d,onchain_defillama_fees_usd_chg_1d,onchain_defillama_fees_usd_chg_7d,onchain_defillama_fees_usd_z_30d
timestamp,,,,,,,,,,,,,,,,,,,,,
2026-06-08,4.198599e+09,5.205459e+09,3.768732e+10,4.925316e+09,5.201669e+10,3.141817e+11,7.224427e+09,56658058.0,0.045713,-0.130067,...,-1.733544,0.000197,-0.011667,-2.194218,0.204065,-0.155942,0.149915,0.145214,-0.014637,0.075704
2026-06-09,4.129998e+09,5.175929e+09,3.750249e+10,4.877264e+09,5.168568e+10,3.147107e+11,6.844394e+09,51492069.0,-0.016339,-0.113027,...,-1.681460,0.001684,-0.010003,-1.778222,-0.052604,-0.269633,-0.051204,-0.091178,-0.176655,-0.708853
2026-06-10,4.061126e+09,5.162583e+09,3.686146e+10,4.781674e+09,5.086684e+10,3.136835e+11,6.709375e+09,53074285.0,-0.016676,-0.073905,...,-1.764953,-0.003264,-0.011104,-1.975732,-0.019727,-0.341014,-0.115433,0.030727,-0.115822,-0.446777
2026-06-11,4.070036e+09,5.156885e+09,3.655583e+10,4.438993e+09,5.022175e+10,3.136407e+11,5.964061e+09,54239853.0,0.002194,-0.036063,...,-1.788952,-0.000136,-0.008434,-1.807584,-0.111085,-0.519760,-0.465818,0.021961,-0.272170,-0.279344
2026-06-12,4.178760e+09,5.210223e+09,3.729126e+10,4.633518e+09,5.131376e+10,3.133113e+11,5.785529e+09,54492695.0,0.026713,-0.002020,...,-1.395088,-0.001050,-0.005862,-1.755887,-0.029935,-0.563699,-0.536923,0.004662,-0.298378,-0.237522


,column,dtype,non_null,missing,missing_pct,sample_value
0,timestamp,datetime64[ms],3700,0,0.00,2016-04-19 00:00:00
1,onchain_defillama_tvl_bitcoin_usd,float64,1911,1789,48.35,64933229.0
2,onchain_defillama_tvl_bsc_usd,float64,2052,1648,44.54,12873962.0
3,onchain_defillama_tvl_ethereum_usd,float64,3181,519,14.03,0.0
4,onchain_defillama_tvl_solana_usd,float64,1914,1786,48.27,148988798.0
5,onchain_defillama_selected_chains_tvl_usd,float64,3181,519,14.03,0.0
6,onchain_defillama_stablecoin_mcap_usd,float64,3118,582,15.73,110105.0
7,onchain_defillama_dex_volume_usd,float64,3692,8,0.22,607.0
8,onchain_defillama_fees_usd,float64,3001,699,18.89,141.0


,timestamp,onchain_defillama_tvl_bitcoin_usd,onchain_defillama_tvl_bsc_usd,onchain_defillama_tvl_ethereum_usd,onchain_defillama_tvl_solana_usd,onchain_defillama_selected_chains_tvl_usd,onchain_defillama_stablecoin_mcap_usd,onchain_defillama_dex_volume_usd,onchain_defillama_fees_usd
0,2026-06-12,4.178760e+09,5.210223e+09,3.729126e+10,4.633518e+09,5.131376e+10,3.133113e+11,5.785529e+09,54492695.0


,column,dtype,non_null,missing,missing_pct,sample_value
0,timestamp,datetime64[ms],3700,0,0.00,2016-04-19 00:00:00
1,onchain_defillama_tvl_bitcoin_usd,float64,1911,1789,48.35,64933229.0
2,onchain_defillama_tvl_bsc_usd,float64,2052,1648,44.54,12873962.0
3,onchain_defillama_tvl_ethereum_usd,float64,3181,519,14.03,0.0
4,onchain_defillama_tvl_solana_usd,float64,1914,1786,48.27,148988798.0
5,onchain_defillama_selected_chains_tvl_usd,float64,3181,519,14.03,0.0
6,onchain_defillama_stablecoin_mcap_usd,float64,3118,582,15.73,110105.0
7,onchain_defillama_dex_volume_usd,float64,3692,8,0.22,607.0
8,onchain_defillama_fees_usd,float64,3001,699,18.89,141.0
9,onchain_defillama_tvl_bitcoin_usd_chg_1d,float64,1910,1790,48.38,0.014002


,timestamp,onchain_defillama_tvl_bitcoin_usd,onchain_defillama_tvl_bsc_usd,onchain_defillama_tvl_ethereum_usd,onchain_defillama_tvl_solana_usd,onchain_defillama_selected_chains_tvl_usd,onchain_defillama_stablecoin_mcap_usd,onchain_defillama_dex_volume_usd,onchain_defillama_fees_usd,onchain_defillama_tvl_bitcoin_usd_chg_1d,...,onchain_defillama_selected_chains_tvl_usd_z_30d,onchain_defillama_stablecoin_mcap_usd_chg_1d,onchain_defillama_stablecoin_mcap_usd_chg_7d,onchain_defillama_stablecoin_mcap_usd_z_30d,onchain_defillama_dex_volume_usd_chg_1d,onchain_defillama_dex_volume_usd_chg_7d,onchain_defillama_dex_volume_usd_z_30d,onchain_defillama_fees_usd_chg_1d,onchain_defillama_fees_usd_chg_7d,onchain_defillama_fees_usd_z_30d
0,2026-06-12,4.178760e+09,5.210223e+09,3.729126e+10,4.633518e+09,5.131376e+10,3.133113e+11,5.785529e+09,54492695.0,0.026713,...,-1.395088,-0.00105,-0.005862,-1.755887,-0.029935,-0.563699,-0.536923,0.004662,-0.298378,-0.237522
